In [8]:
from pathlib import Path

direc=Path().absolute()

input_path= {
    "a": Path.joinpath(direc,"data/a_example"),
    "b": Path.joinpath(direc,"data/b_little_bit_of_everything.in"),
    "c": Path.joinpath(direc,"data/c_many_ingredients.in"),
    "d": Path.joinpath(direc,"data/d_many_pizzas.in"),
    "e": Path.joinpath(direc,"data/e_many_teams.in")
}

output_path = {
    "a": Path.joinpath(direc,"out/a_example.txt"),
    "b": Path.joinpath(direc,"out/b_little_bit_of_everything.txt"),
    "c": Path.joinpath(direc,"out/c_many_ingredients.txt"),
    "d": Path.joinpath(direc,"out/d_many_pizzas.txt"),
    "e": Path.joinpath(direc,"out/e_many_teams.txt")
}

print("Welcome to Group G's implementation of ""Even More Pizzas""\n")
print("The following dataset options are available: a, b, c, d and e \n")

data=""
while data not in ["a","b","c","d","e"]:
    data = input("Please select your data: \n")
    if (data not in ["a","b","c","d","e"]):
        print("That is not an available dataset.\n")

pizza_list=[]
teams={}
input_loc=input_path[data]
output_loc=output_path[data]

with open(input_loc) as file:
        for line_index, line in enumerate(file):
            actual_line = line.strip().split()
            if line_index == 0:
                teams = {
                    2: int(actual_line[1]),
                    3: int(actual_line[2]),
                    4: int(actual_line[3])
                }
            else:
                pizza = {
                "id": line_index - 1,
                "n_ingredients": int(actual_line[0]),
                "ingredients": actual_line[1:]
            }
                pizza_list.append(pizza)

#print(pizza_list[0]['ingredients'])

Welcome to Group G's implementation of Even More Pizzas

The following dataset options are available: a, b, c, d and e 



In [3]:
def solve_pizza_problem(pizzas, t2, t3, t4):
    # Sort pizzas by number of ingredients (descending) to start with 'richer' options
    pizzas.sort(key=lambda x: len(x['ingredients']), reverse=True)
    
    deliveries = []
    used_pizzas = [False] * len(pizzas)
    
    # Process teams from largest to smallest (4 -> 3 -> 2) 
    # because squares of larger numbers yield higher scores.
    for team_size, team_count in [(4, t4), (3, t3), (2, t2)]:
        for _ in range(team_count):
            current_team_pizzas = []
            current_ingredients = set()
            
            # Fill the team requirements
            for _ in range(team_size):
                best_pizza_idx = -1
                max_new_ingredients = -1
                
                # Look for the pizza that adds the most value
                for i in range(len(pizzas)):
                    if not used_pizzas[i]:
                        # Calculate how many NEW ingredients this pizza adds
                        new_count = len(set(pizzas[i]['ingredients']) - current_ingredients)
                        
                        if new_count > max_new_ingredients:
                            max_new_ingredients = new_count
                            best_pizza_idx = i
                        
                        # Optimization: if a pizza adds all its ingredients as new, 
                        # it's a strong candidate; we can break early in large datasets.
                
                if best_pizza_idx != -1:
                    used_pizzas[best_pizza_idx] = True
                    current_team_pizzas.append(pizzas[best_pizza_idx]['id'])
                    current_ingredients.update(pizzas[best_pizza_idx]['ingredients'])
                else:
                    # Not enough pizzas left to fill this team
                    break
            
            if len(current_team_pizzas) == team_size:
                deliveries.append((team_size, current_team_pizzas))
            else:
                # Backtrack: if team wasn't filled, mark pizzas as available again
                for p_id in current_team_pizzas:
                    used_pizzas[p_id] = False
                    
    return deliveries

In [4]:
def score(pizzas,deliveries):
    score_list=[]
    for i in range(len(deliveries)):
        delivered_pizzas=deliveries[i][1]
        ingredients=[]
        for id in delivered_pizzas:
            ingredient_list=pizzas[id]['ingredients']
            for ingredient in ingredient_list:
                ingredients.append(ingredient)
        unique_ingredients=set(ingredients)
        score_list.append(len(unique_ingredients)**2)
        #print(score_list)
    if len(deliveries)==1:
        total_score=score_list[0]
    else:
        total_score=sum(score_list)
    return total_score

In [5]:
def output_write(output_path,deliveries):
    with open(output_path, "w") as file:
        file.write(str(len(deliveries)))
        file.write("\n")
        for team_size, pizza_id in deliveries:
            line = " ".join(map(str, [team_size] + pizza_id))
            file.write(line + "\n")

In [9]:
deliveries=solve_pizza_problem(pizza_list, teams[2], teams[3], teams[4])
sco=score(pizza_list,deliveries)
print(deliveries)
print(sco)
output_write(output_loc,deliveries)

[(4, [0, 1, 2, 3])]
49


In [ ]:
''' Local
a: 49 (< 1 segundo)
b: 6502 (< 1 segundo)
c: 229352265 (3-4 minutos)
d: 2046959 (30-40 minutos)
e: 8257534 (1 hora)
'''